In [4]:
import pandas as pd
import re
from google import genai
from dotenv import load_dotenv
import json
import requests
import re
import ast
categorization_sys_prompt = '''
You are a financial transaction categorization engine. You will receive a Python list of merchant names from bank transactions and classify each one into the correct spending category.

## CATEGORIES
Classify each merchant into exactly one of the following categories:
- `Groceries` - supermarkets, food stores, fresh markets
- `Dining` - restaurants, cafes, fast food, takeaway, food delivery apps
- `Fuel` - petrol stations, gas stations, merchants ending with "gazi"
- `Transport` - public transport, taxis, ride-hailing (Uber, Bolt), parking
- `Travel` - flights, hotels, booking platforms, car rentals, travel agencies
- `Pharmacy` - pharmacies, drugstores, medical supplies
- `Healthcare` - clinics, hospitals, doctors, dentists, labs
- `Entertainment` - cinemas, concerts, events, gaming, streaming services
- `Shopping` - clothing, electronics, home goods, online marketplaces
- `Subscriptions` - recurring software, SaaS, memberships, apps
- `Utilities` - electricity, water, gas, internet, phone bills
- `Education` - courses, books, tuition, e-learning platforms
- `Fitness` - gyms, sports equipment, wellness apps
- `PersonalCare` - salons, barbers, beauty products, spas
- `Finance` - bank fees, insurance, investments, loan repayments
- `Other` - anything that does not clearly fit the above

## RULES
1. Use your real-world knowledge of brands, chains, and services to classify.
2. If a name is ambiguous, pick the most statistically likely category for that merchant type.
3. Never leave a category blank — always assign the best possible match.
4. If genuinely uncertain, use `Other`.
5. Do NOT invent new categories beyond the list above.

## OUTPUT FORMAT
Return ONLY a valid Python dictionary — no explanation, no markdown, no code fences.
Exact structure:

{
  "results": [
    {"merchant": "<original name>", "category": "<category>", "confidence": "<high|medium|low>"},
    ...
  ]
}

## INPUT
'''

load_dotenv('../../backend/.env')
client = genai.Client()
def get_payments_df(transactions_df:pd.DataFrame) -> pd.DataFrame:
    """separates actual payments from just transactions"""
    def find_transaction_object(text):
        pattern = r"ობიექტი:\s*([^,]+)"
        res = re.search(pattern, text)
        if res:
            return res.group(1)
    
    payments_df = transactions_df.loc[transactions_df["transaction_type"] == "გადახდა"] # separate actal payments from transactions
    payments_df["transaction_object"] = payments_df["დანიშნულება"].apply(lambda x:find_transaction_object(x) ) # separate where transaction was actually performed
    payments_df["transaction_object"] = payments_df["transaction_object"].fillna(value="gadakhda") # fill wherever object was unavailable
    for currency in ["GEL","USD","EUR","GBP"]:
        payments_df[currency] = payments_df[currency].apply(lambda x: x * -1)
    payments_df.drop(columns=["transaction_type"],inplace=True)
    payments_df.fillna(value=0, inplace=True) # fill other NaN values
    payments_df["transaction_object"] = payments_df["transaction_object"].apply(lambda x: x.lower()) # convert to lowercase
    

    return payments_df

def get_spending_by_category(df:pd.DataFrame):
    df = df.groupby(["category"])[["GEL","USD","EUR","GBP"]].sum().sort_values(by="GEL", ascending=False)
    return df


def clean_response(text, parse_as="json"):
    """
    Cleans LLM output by removing code fences and parses it.
    
    Args:
        text (str): The LLM response text.
        parse_as (str): "json" or "python". Determines parser.
        
    Returns:
        Parsed Python object (dict/list/etc.).
    """
    # Remove ```json, ```python, or ``` code fences
    cleaned = re.sub(r'```(?:json|python)?', '', text).strip()
    
    if parse_as == "json":
        return json.loads(cleaned)
    elif parse_as == "python":
        return ast.literal_eval(cleaned)
    else:
        raise ValueError("parse_as must be 'json' or 'python'")

def parse_categorization_response(response):
    try:
        response_text = response.text
        response_text = clean_response(response_text, parse_as="json")
        # print(response_text)
        categories_df = pd.DataFrame(response_text["results"])
    except Exception as e:
        print(f"Error parsing categorization response: {e}")
        return None

    return categories_df

def get_unknown_merchants(merchants_df:pd.DataFrame, merchants_list:list):
    if merchants_df.empty:
        return merchants_list
    existing_merchants = list(merchants_df["merchant"].unique())
    merchants_list = [merchant for merchant in merchants_list if merchant not in existing_merchants]
    return merchants_list 

def run_llm_categorization(merchants, categorization_sys_prompt ):
    merchants_formatted = "\n".join(f"{i+1}. {m}" for i, m in enumerate(merchants))

    
    config = genai.types.GenerateContentConfig(
            system_instruction=categorization_sys_prompt
        )
    response = client.models.generate_content(model="gemini-3-flash-preview",
                                              contents=merchants_formatted,
                                              config=config)
    return response
def get_eixisting_categories_df(df:pd.DataFrame):
 
    # Fetch categories from FastAPI server
    response = requests.get("http://localhost:8000/categories")
    if response.status_code == 200:
        categories_table = pd.DataFrame(response.json())
    else:
        print("Failed to fetch categories from server:", response.status_code)
        categories_table = pd.DataFrame(columns=["merchant","category","confidence"])  # empty DataFrame fallback
    return categories_table

In [5]:

transactions_df = pd.read_excel("../../data/raw/amonaweri.xlsx", sheet_name="ტრანზაქციები")
transactions_df.drop(columns=["Unnamed: 2"],inplace=True) # drop unneccessary col
transactions_df["transaction_type"] = transactions_df["დანიშნულება"].apply(lambda x: x.split()[0]) # separate by transaction type
transactions_df["თარიღი"] = pd.to_datetime(transactions_df["თარიღი"], dayfirst=True) # convert to datetime for simplicity

df = get_payments_df(transactions_df)
df

c:\Users\Saba\.conda\envs\financeAI\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,თარიღი,დანიშნულება,GEL,USD,EUR,GBP,transaction_object
2,2025-03-07,"გადახდა - თანხა: GEL2.94; ობიექტი: LIBRE, Tbil...",2.94,0.0,0.0,0.0,libre
6,2025-03-09,"გადახდა - თანხა GEL3.00; გადახდა, 09/03/2025 ,...",3.00,0.0,0.0,0.0,gadakhda
12,2025-03-09,"გადახდა - თანხა: GEL5.34; ობიექტი: Nikora, Tbi...",5.34,0.0,0.0,0.0,nikora
13,2025-03-09,"გადახდა - თანხა: GEL68.06; ობიექტი: libre, Tbi...",68.06,0.0,0.0,0.0,libre
14,2025-03-10,გადახდა - თანხა: GEL6.30; ობიექტი: shps gama 2...,6.30,0.0,0.0,0.0,shps gama 2023
...,...,...,...,...,...,...,...
500,2026-03-04,"გადახდა - თანხა: GEL38.97; ობიექტი: XS Toys, T...",38.97,0.0,0.0,0.0,xs toys
501,2026-03-05,"გადახდა - თანხა: GEL3.20; ობიექტი: MAGNITI, Tb...",3.20,0.0,0.0,0.0,magniti
503,2026-03-05,"გადახდა - თანხა GEL7.00; გადახდა, 05/03/2026 ,...",7.00,0.0,0.0,0.0,gadakhda
505,2026-03-05,"გადახდა - თანხა: GEL8.20; ობიექტი: ORI NABIJI,...",8.20,0.0,0.0,0.0,ori nabiji


# spent so far

In [29]:
from datetime import datetime

currency = "GEL"
today = datetime.today()
start_of_month = today.replace(day=1)

# Current month spending
current_month_df = df[df['თარიღი'] >= start_of_month]
spent_so_far = 300 #current_month_df[currency].sum()
days_passed = today.day

def get_spending_by_month(df:pd.DataFrame):
    df["month"] = df["თარიღი"].dt.month
    df["year"] = df["თარიღი"].dt.year
    
    df = df.groupby(["year","month"])[["GEL","USD","EUR","GBP"]].sum()
    return df

median_monthly_spending = get_spending_by_month(df).reset_index()[currency].median()

percentage = spent_so_far / median_monthly_spending * 100
expected_percentage = days_passed / 30 * 100
predicted_total = spent_so_far / days_passed * 30
if percentage > expected_percentage * 1.3:
    print("You’re spending faster than usual. At this pace, you may exceed your typical monthly spending.")
else:
    print(f"You spent {spent_so_far:.0f} {currency} in the first {days_passed} days. That's {percentage:.0f}% of your usual monthly spending.")
print(f"At this pace, you’re projected to spend {predicted_total:.0f} {currency} this month.")



You spent 300 GEL in the first 24 days. That's 91% of your usual monthly spending.
At this pace, you’re projected to spend 375 GEL this month.


In [ ]:
#  --------------------- subscriptions code ----------------------
# from datetime import datetime
# for currency in ["GEL","USD","EUR","GBP"]:
#     grouped = df.groupby(["transaction_object", currency])
#     for (merchant, currency), group in grouped:
        
        
#         # for i,row in group.iterrows():
#         #     print(row["თარიღი"])
#         if currency != 0.0:
#             dates_list = group["თარიღი"].tolist()
#             if len(dates_list) > 1:
#                 dates_list = [datetime.strptime(date, "%d/%m/%Y") for date in dates_list]
#                 diffs = []
#                 for i in range(len(dates_list)-1):
#                     diff = dates_list[i+1] - dates_list[i]
#                     diffs.append(diff.days)
#                 print("Merchant:", merchant)
#                 print("Currency:", currency)
#                 print(dates_list)
#                 print(diffs)
#                 print("-" * 40)

Merchant: agrohub
Currency: 6.6
[datetime.datetime(2025, 4, 23, 0, 0), datetime.datetime(2025, 4, 28, 0, 0)]
[5]
----------------------------------------
Merchant: carrefour
Currency: 1.75
[datetime.datetime(2025, 4, 6, 0, 0), datetime.datetime(2025, 8, 28, 0, 0)]
[144]
----------------------------------------
Merchant: carrefour
Currency: 6.2
[datetime.datetime(2025, 6, 28, 0, 0), datetime.datetime(2025, 7, 15, 0, 0)]
[17]
----------------------------------------
Merchant: gadakhda
Currency: 1.0
[datetime.datetime(2025, 3, 29, 0, 0), datetime.datetime(2025, 5, 7, 0, 0), datetime.datetime(2025, 5, 27, 0, 0), datetime.datetime(2025, 6, 10, 0, 0), datetime.datetime(2025, 6, 27, 0, 0), datetime.datetime(2025, 7, 12, 0, 0), datetime.datetime(2025, 7, 28, 0, 0), datetime.datetime(2026, 2, 28, 0, 0)]
[39, 20, 14, 17, 15, 16, 215]
----------------------------------------
Merchant: gadakhda
Currency: 2.0
[datetime.datetime(2025, 3, 28, 0, 0), datetime.datetime(2025, 4, 28, 0, 0), datetime.date

In [ ]:
def categorize_merchants_pipeline(df:pd.DataFrame):
    existing_categories_df = get_eixisting_categories_df(df) # fetch existing categories from server

    merchants_list = list(df["transaction_object"].unique()) # get all merchants from transactions

    unknown_merchants = get_unknown_merchants(existing_categories_df, merchants_list) # get whicih merchants have unassigned categories

    llm_categories_response = run_llm_categorization(unknown_merchants, categorization_sys_prompt) # run llm to assign categories
    categories_df = parse_categorization_response(llm_categories_response) # parse llm response 

    records = categories_df[["merchant", "category", "confidence"]].to_dict(orient="records") # convert to records
    response = requests.post("http://localhost:8000/upload-categories", json=records) # upload to server

    if response.status_code == 200:
        print("Categories uploaded successfully")
    else:
        print("Failed to upload categories:", response.status_code)
    


['libre',
 'gadakhda',
 'nikora',
 'shps gama 2023',
 'spar',
 'magniti',
 'ori nabiji',
 'shemogevle',
 'ie goderdzi apkhazashvili',
 'tea house veris bagi',
 'pablisit service',
 'agrohub',
 'carrefour',
 "mcdonald's",
 'pipes',
 'bunebrivi gazi',
 'tsiskvili market',
 'neogas',
 'gama',
 'wolt',
 'paiberi',
 'oneprice',
 'i/m zaira chichinadze',
 'billiard hall',
 'georgian post',
 'vip pay',
 'yandex.go',
 'ji ji house 3',
 'ltd build more',
 'pegasus pegasus',
 'piatto',
 'evex',
 'zgapari',
 'ltd mp development',
 'omega gazi',
 'pharmadepot',
 'petra beach',
 'wissol',
 'socar',
 'daily 157',
 'daily',
 'psp',
 'aversi 95',
 'market spar',
 'xs toys']

In [162]:
keys = merchants_df["merchant"]
values = merchants_df["category"]

merchants_dict = dict(zip(keys,values))
df["category"] = df["transaction_object"].map(merchants_dict)
df["თარიღი"] = pd.to_datetime(df["თარიღი"], dayfirst=True) 
df

,თარიღი,დანიშნულება,GEL,USD,EUR,GBP,transaction_object,category
2,2025-03-07,"გადახდა - თანხა: GEL2.94; ობიექტი: LIBRE, Tbil...",2.94,0.0,0.0,0.0,libre,Groceries
6,2025-03-09,"გადახდა - თანხა GEL3.00; გადახდა, 09/03/2025 ,...",3.00,0.0,0.0,0.0,gadakhda,Utilities
12,2025-03-09,"გადახდა - თანხა: GEL5.34; ობიექტი: Nikora, Tbi...",5.34,0.0,0.0,0.0,nikora,Groceries
13,2025-03-09,"გადახდა - თანხა: GEL68.06; ობიექტი: libre, Tbi...",68.06,0.0,0.0,0.0,libre,Groceries
14,2025-03-10,გადახდა - თანხა: GEL6.30; ობიექტი: shps gama 2...,6.30,0.0,0.0,0.0,shps gama 2023,Healthcare
...,...,...,...,...,...,...,...,...
500,2026-03-04,"გადახდა - თანხა: GEL38.97; ობიექტი: XS Toys, T...",38.97,0.0,0.0,0.0,xs toys,Shopping
501,2026-03-05,"გადახდა - თანხა: GEL3.20; ობიექტი: MAGNITI, Tb...",3.20,0.0,0.0,0.0,magniti,Groceries
503,2026-03-05,"გადახდა - თანხა GEL7.00; გადახდა, 05/03/2026 ,...",7.00,0.0,0.0,0.0,gadakhda,Utilities
505,2026-03-05,"გადახდა - თანხა: GEL8.20; ობიექტი: ORI NABIJI,...",8.20,0.0,0.0,0.0,ori nabiji,Groceries


In [ ]:
# def get_spending_by_month(df:pd.DataFrame):
#     df["month"] = df["თარიღი"].dt.month
#     df["year"] = df["თარიღი"].dt.year
    
#     df = df.groupby(["month","year"])[["GEL","USD","EUR","GBP"]].sum()
#     return df
def get_spending_by_category(df:pd.DataFrame):
    df = df.groupby(["category"])[["GEL","USD","EUR","GBP"]].sum().sort_values(by="GEL", ascending=False)
    return df

# def get_total_spending(df:pd.DataFrame):
#     total = df[["GEL","USD","EUR","GBP"]].sum()
#     return total

# def get_transaction_means(df:pd.DataFrame):
#     means = df[["GEL","USD","EUR","GBP"]].mean()
#     return means
# def get_transaction_medians(df:pd.DataFrame):
#     #TODO: make it ignore zeros
#     medians = df[["GEL","USD","EUR","GBP"]].median()
#     return medians

# def get_biggest_spending(df:pd.DataFrame,currency:str="GEL"):
#     return df.sort_values(by=currency, ascending=False).head(1)
# def get_transaction_count(df:pd.DataFrame):
#     count = int(df['თარიღი'].count())
#     return count

# def get_spending_by_merchant(df:pd.DataFrame):
#     df = df.groupby(["transaction_object"])[["GEL","USD","EUR","GBP"]].sum().sort_values(by="GEL", ascending=False)
#     return df


# def get_transactions_per_day(df:pd.DataFrame):
#     return df.groupby(["თარიღი"])[["GEL"]].count().sort_values(by="GEL", ascending=False)

# def get_most_active_day(df:pd.DataFrame):
#     return get_transactions_per_day(df).head(1)

# def get_anomaly_transactions(df:pd.DataFrame, currency:str="GEL"):
#     std = df[currency].std()
#     mean = df[currency].mean()
#     df_anomaly = df[df[currency] > mean + 2 * std]
#     return df_anomaly
    
# def get_month_comparison_analysis(latest_month_df, previous_month_biggest_spending_category, currency="GEL"):
#     if len(previous_month_biggest_spending_category) != 0:
#         percent_difference = ((latest_month_df[currency] - previous_month_biggest_spending_category[currency]) / previous_month_biggest_spending_category[currency]) * 100 if previous_month_biggest_spending_category[currency] != 0 else float('inf')
        
#         if percent_difference > 0:
#             return f"This month your spending on {latest_month_df["category"]} was more than previous month by {percent_difference}%"
#         else:
#             return f"This month your spending on {latest_month_df["category"]} was less than previous month by {percent_difference}%"
#     else:
#         return "No previous month data available"

# def get_month_category_comparison(df:pd.DataFrame, currency:str="GEL"):
#     df["month"] = df["თარიღი"].dt.month
#     df["year"] = df["თარიღი"].dt.year
#     df = df.groupby(["year","month","category"])[["GEL","USD","EUR","GBP"]].sum()

#     df_reset = df.reset_index()

#     latest = df_reset.sort_values(by=["year",'month']).iloc[-1]
#     latest_month, latest_year = latest[["month","year"]]

#     previous_month = latest_month - 1 if latest_month != 1 else 12

#     latest_month_df = df_reset[(df_reset["year"] == latest_year) & (df_reset['month'] == latest_month)]
#     previous_month_df = df_reset[(df_reset["year"] == latest_year) & (df_reset['month'] == previous_month)]

#     latest_month_df = latest_month_df.sort_values(by=[currency], ascending=False).head(1)
#     previous_month_df = previous_month_df.sort_values(by=[currency], ascending=False)
#     biggest_spending_category = latest_month_df["category"].values[0]
#     previous_month_biggest_spending_category = previous_month_df[previous_month_df["category"] == biggest_spending_category].values
#     return get_month_comparison_analysis(latest_month_df, previous_month_biggest_spending_category, currency)


    

'No previous month data available'

Percent difference: 73.53%


,year,month,category,GEL,USD,EUR,GBP
42,2026,3,Shopping,118.97,0.0,0.0,0.0


,year,month,category,GEL,USD,EUR,GBP
